This model reads a piece of English text and picks out the mountain names
in it - from famous peaks like Everest to lesser-known ones tucked into a
sentence.

*   **Model:** `microsoft/deberta-v3-base`, fine-tuned for 2 epochs on the binary labeling scheme (O / MOUNTAIN).
*   **Test set results:** precision 0.833, recall 0.853, F1 0.843
* **Model weights:** [HuggingFace](huggingface.co/vl00835/mountain-ner-deberta-v3-base-binary)

The inference logic is imported from `inference.py` rather than duplicated
here, so this notebook demonstrates exactly the code that ships with the
project.

In [3]:
!pip install spacy -q

In [4]:
import sys
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/mountain_ner"
sys.path.append(PROJECT_DIR)

from inference import MountainNER

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
MODEL_PATH = "vl00835/mountain-ner-deberta-v3-base-binary"

ner = MountainNER(MODEL_PATH)

print(f"Model:  {MODEL_PATH}")
print(f"Device: {ner.device}")
print(f"Labels: {ner.id2label}")

model.safetensors: reconstructing file:   0%|          |  0.00B /  735MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Model:  vl00835/mountain-ner-deberta-v3-base-binary
Device: cpu
Labels: {0: 'O', 1: 'MOUNTAIN'}


In [6]:
text = "We climbed Mount Everest last summer and later visited Kilimanjaro in Tanzania."

for entity in ner.extract_mountains(text):
    print(f"  {entity['name']}  (confidence {entity['confidence']:.3f})")

  Mount Everest  (confidence 0.980)
  Kilimanjaro  (confidence 0.904)


In [7]:
from spacy import displacy

def show_entities(text, ner=ner):
    """Render model predictions with displaCy."""
    ents = []
    offset = 0
    for word, label, _ in ner.predict_words(text):
        if label != "O":
            ents.append({
                "start": offset,
                "end": offset + len(word),
                "label": "MOUNTAIN",
            })
        offset += len(word) + 1        # +1 for the space between words

    displacy.render(
        {"text": text, "ents": ents},
        style="ent",
        manual=True,
        jupyter=True,
    )

In [8]:
show_entities(text)

In [9]:
examples = [
    "The Alps stretch across eight countries in Europe.",
    "K2 is considered more dangerous to climb than Everest.",
    "The summit of Ben Nevis is often covered in clouds.",
    "Mont Blanc and the Matterhorn are both in the Alps.",
    "She works as a data scientist in Kyiv and enjoys hiking.",
]

for example in examples:
    show_entities(example)

In [10]:
paragraph = (
    "The Himalayas form a natural barrier between the Indian subcontinent and "
    "the Tibetan Plateau. Mount Everest, the highest peak on Earth, sits on the "
    "border between Nepal and China, while Annapurna and Dhaulagiri rise nearby. "
    "Further west, Nanga Parbat is known among climbers as one of the most "
    "difficult ascents in the world. The range was formed by the collision of "
    "tectonic plates that began roughly fifty million years ago."
)

show_entities(paragraph)

In [11]:
import pandas as pd

entities = ner.extract_mountains(paragraph)

pd.DataFrame(entities).sort_values("confidence", ascending=False).reset_index(drop=True)

,name,confidence
0,Dhaulagiri,0.9852
1,Mount Everest,0.9839
2,Nanga Parbat,0.9822
3,Himalayas,0.9513


In [12]:
edge_cases = [
    # geographic proper nouns that are not mountains
    "She works as a data scientist in Kyiv and enjoys hiking.",
    "The Amazon river flows through Brazil, Peru and Colombia.",

    # generic nouns that appear as mountain mentions in the training data
    "They reached the summit shortly after sunrise.",

    # the same name with and without attached punctuation
    "Everest is the highest peak.",
    "The highest peak is Everest.",
]

for case in edge_cases:
    show_entities(case)

# Summary

**What works reliably**
* Well-known single- and multi-word mountain names (Mount Everest, Ben Nevis,
Nanga Parbat), including names the model is less familiar with.
* Mountain ranges and massifs (the Alps, the Himalayas).
* Rejecting non-mountain geographic proper nouns — cities (Kyiv) and rivers
(the Amazon) are correctly left untagged.

**Known limitations**
- **Missed entities:** rare names in enumerations can be skipped - Annapurna was not detected in the paragraph above, while Dhaulagiri from the same sentence was found with the highest confidence in the text.
- **Generic nouns:** "summit" is tagged as a mountain even when no specific mountain is mentioned. This traces directly back to the training data, where generic geographic nouns (summit, peak, ridge) are sometimes labeled as mountain entities.
- **Attached punctuation:** the dataset was tokenized with a whitespace split, so the model treats "Everest." as a separate token from "Everest". Punctuation
is stripped in post-processing, but the underlying token mismatch remains.